## Introduction to NLP with PySpark

### NLP Tools

1. Tokenizer
2. Stop word removal
3. n-grams
4. Term Frequency - Inverse Document Frequency (TF-IDF)
5. Count Vectorizer

### SMS Spam Collection

* Download the data for later usage.

In [1]:
# !curl https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip >> smsspamcollection.zip

In [2]:
# !tar -xf smsspamcollection.zip

In [3]:
# import os
# import sys

# os.environ["PYSPARK_PYTHON"] = sys.executable
# os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.appName('NLP').getOrCreate()

D:\miniconda3\envs\pyspark_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


#### Tokenizer

In [6]:
from pyspark.ml.feature import Tokenizer, RegexTokenizer
from pyspark.sql.functions import col, udf
from pyspark.sql.types import IntegerType

In [7]:
sent_df = spark.createDataFrame(
    [
        (0, "Hello, I am happy to be learning Apache Spark"),
        (1, "I enjoy learning about Python and SQL programming"),
        (2, "I am familiar with machine learning applications."),
        (3, "here,is,a,list,of,words")
    ],
    ['id', 'sentence']
)

In [8]:
sent_df.show()

+---+--------------------+
| id|            sentence|
+---+--------------------+
|  0|Hello, I am happy...|
|  1|I enjoy learning ...|
|  2|I am familiar wit...|
|  3|here,is,a,list,of...|
+---+--------------------+



In [9]:
tokenizer = Tokenizer(inputCol='sentence', outputCol='words')

regexTokenizer = RegexTokenizer(inputCol='sentence', outputCol='words', pattern='\\W')

countTokens = udf(lambda w: len(w), IntegerType())

D:\miniconda3\envs\pyspark_env\Lib\site-packages\pyspark\sql\udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [10]:
tokenized = tokenizer.transform(sent_df)

In [11]:
tokenized.show()

+---+--------------------+--------------------+
| id|            sentence|               words|
+---+--------------------+--------------------+
|  0|Hello, I am happy...|[hello,, i, am, h...|
|  1|I enjoy learning ...|[i, enjoy, learni...|
|  2|I am familiar wit...|[i, am, familiar,...|
|  3|here,is,a,list,of...|[here,is,a,list,o...|
+---+--------------------+--------------------+



In [12]:
tokenized.select("sentence", "words").withColumn("tokens", countTokens(col('words'))).show()

+--------------------+--------------------+------+
|            sentence|               words|tokens|
+--------------------+--------------------+------+
|Hello, I am happy...|[hello,, i, am, h...|     9|
|I enjoy learning ...|[i, enjoy, learni...|     8|
|I am familiar wit...|[i, am, familiar,...|     7|
|here,is,a,list,of...|[here,is,a,list,o...|     1|
+--------------------+--------------------+------+



In [13]:
regexTokenized = regexTokenizer.transform(sent_df)
regexTokenized.select("sentence", "words").withColumn("tokens", countTokens(col('words'))).show()

+--------------------+--------------------+------+
|            sentence|               words|tokens|
+--------------------+--------------------+------+
|Hello, I am happy...|[hello, i, am, ha...|     9|
|I enjoy learning ...|[i, enjoy, learni...|     8|
|I am familiar wit...|[i, am, familiar,...|     7|
|here,is,a,list,of...|[here, is, a, lis...|     6|
+--------------------+--------------------+------+



In [18]:
sent_df_token = regexTokenized.select("sentence", "words").withColumn("tokens", countTokens(col('words')))

#### Stop Word Removal

In [14]:
from pyspark.ml.feature import StopWordsRemover

In [15]:
sent_df.show(truncate=False)

+---+-------------------------------------------------+
|id |sentence                                         |
+---+-------------------------------------------------+
|0  |Hello, I am happy to be learning Apache Spark    |
|1  |I enjoy learning about Python and SQL programming|
|2  |I am familiar with machine learning applications.|
|3  |here,is,a,list,of,words                          |
+---+-------------------------------------------------+



In [20]:
remover = StopWordsRemover(inputCol="words", outputCol="cleaned")

In [21]:
remover.transform(sent_df_token).show(truncate=False)

+-------------------------------------------------+----------------------------------------------------------+------+-------------------------------------------+
|sentence                                         |words                                                     |tokens|cleaned                                    |
+-------------------------------------------------+----------------------------------------------------------+------+-------------------------------------------+
|Hello, I am happy to be learning Apache Spark    |[hello, i, am, happy, to, be, learning, apache, spark]    |9     |[hello, happy, learning, apache, spark]    |
|I enjoy learning about Python and SQL programming|[i, enjoy, learning, about, python, and, sql, programming]|8     |[enjoy, learning, python, sql, programming]|
|I am familiar with machine learning applications.|[i, am, familiar, with, machine, learning, applications]  |7     |[familiar, machine, learning, applications]|
|here,is,a,list,of,words    

#### n-grams

In [22]:
from pyspark.ml.feature import NGram

In [24]:
sent_df_token.show()

+--------------------+--------------------+------+
|            sentence|               words|tokens|
+--------------------+--------------------+------+
|Hello, I am happy...|[hello, i, am, ha...|     9|
|I enjoy learning ...|[i, enjoy, learni...|     8|
|I am familiar wit...|[i, am, familiar,...|     7|
|here,is,a,list,of...|[here, is, a, lis...|     6|
+--------------------+--------------------+------+



In [27]:
bigrams = NGram(n=2, inputCol='words', outputCol='bigrams')

In [28]:
bigram_df = bigrams.transform(sent_df_token)

In [30]:
bigram_df.select("bigrams").show(truncate=False)

+---------------------------------------------------------------------------------------------+
|bigrams                                                                                      |
+---------------------------------------------------------------------------------------------+
|[hello i, i am, am happy, happy to, to be, be learning, learning apache, apache spark]       |
|[i enjoy, enjoy learning, learning about, about python, python and, and sql, sql programming]|
|[i am, am familiar, familiar with, with machine, machine learning, learning applications]    |
|[here is, is a, a list, list of, of words]                                                   |
+---------------------------------------------------------------------------------------------+



#### Term Frequency - Inverse Document Frequency (TF-IDF)

In [31]:
from pyspark.ml.feature import HashingTF, IDF, Tokenizer

In [34]:
sent_df = spark.createDataFrame(
    [
        (0, 0.0, "Hello, I am happy to be learning Apache Spark"),
        (1, 0.0, "I enjoy learning about Python and SQL programming"),
        (2, 1.0, "I am familiar with machine learning applications."),
        (3, 1.0, "here,is,a,list,of,words")
    ],
    ['id', 'label', 'sentence']
)

In [35]:
sent_df.show(truncate=False)

+---+-----+-------------------------------------------------+
|id |label|sentence                                         |
+---+-----+-------------------------------------------------+
|0  |0.0  |Hello, I am happy to be learning Apache Spark    |
|1  |0.0  |I enjoy learning about Python and SQL programming|
|2  |1.0  |I am familiar with machine learning applications.|
|3  |1.0  |here,is,a,list,of,words                          |
+---+-----+-------------------------------------------------+



In [36]:
tokenizer = RegexTokenizer(inputCol='sentence', outputCol='words', pattern='\\W')
words_df = tokenizer.transform(sent_df)

In [37]:
words_df.show(truncate=False)

+---+-----+-------------------------------------------------+----------------------------------------------------------+
|id |label|sentence                                         |words                                                     |
+---+-----+-------------------------------------------------+----------------------------------------------------------+
|0  |0.0  |Hello, I am happy to be learning Apache Spark    |[hello, i, am, happy, to, be, learning, apache, spark]    |
|1  |0.0  |I enjoy learning about Python and SQL programming|[i, enjoy, learning, about, python, and, sql, programming]|
|2  |1.0  |I am familiar with machine learning applications.|[i, am, familiar, with, machine, learning, applications]  |
|3  |1.0  |here,is,a,list,of,words                          |[here, is, a, list, of, words]                            |
+---+-----+-------------------------------------------------+----------------------------------------------------------+



In [38]:
hashingTF = HashingTF(inputCol='words', outputCol='rawFeatures', numFeatures=20)

featurized = hashingTF.transform(words_df)

In [39]:
featurized.show(truncate=False)

+---+-----+-------------------------------------------------+----------------------------------------------------------+-----------------------------------------------------------------+
|id |label|sentence                                         |words                                                     |rawFeatures                                                      |
+---+-----+-------------------------------------------------+----------------------------------------------------------+-----------------------------------------------------------------+
|0  |0.0  |Hello, I am happy to be learning Apache Spark    |[hello, i, am, happy, to, be, learning, apache, spark]    |(20,[3,5,6,7,8,9,12,15,16],[1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|1  |0.0  |I enjoy learning about Python and SQL programming|[i, enjoy, learning, about, python, and, sql, programming]|(20,[1,5,9,11,12,16],[1.0,1.0,1.0,1.0,2.0,2.0])                  |
|2  |1.0  |I am familiar with machine learning applications.|[i, 

In [40]:
idf = IDF(inputCol='rawFeatures', outputCol='features')
idf_model = idf.fit(featurized)

In [41]:
rescale = idf_model.transform(featurized)
rescale.select('label', 'features').show()

+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|(20,[3,5,6,7,8,9,...|
|  0.0|(20,[1,5,9,11,12,...|
|  1.0|(20,[0,2,5,10,12,...|
|  1.0|(20,[7,8,9,12,15]...|
+-----+--------------------+



#### Count Vectorization

In [42]:
from pyspark.ml.feature import CountVectorizer

In [43]:
df = spark.createDataFrame([
    (0, list("abcde")),
    (1, list("abbbccdee"))
], ['id', 'words'])

In [44]:
df.show()

+---+--------------------+
| id|               words|
+---+--------------------+
|  0|     [a, b, c, d, e]|
|  1|[a, b, b, b, c, c...|
+---+--------------------+



In [45]:
cv = CountVectorizer(inputCol='words', outputCol='features', vocabSize=5, minDF=2.0)

In [46]:
model = cv.fit(df)

In [47]:
res = model.transform(df)
res.show(truncate=False)

+---+---------------------------+-------------------------------------+
|id |words                      |features                             |
+---+---------------------------+-------------------------------------+
|0  |[a, b, c, d, e]            |(5,[0,1,2,3,4],[1.0,1.0,1.0,1.0,1.0])|
|1  |[a, b, b, b, c, c, d, e, e]|(5,[0,1,2,3,4],[3.0,2.0,2.0,1.0,1.0])|
+---+---------------------------+-------------------------------------+

